In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
import pulp

os.makedirs('outputs', exist_ok=True)
pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (10, 4)
np.random.seed(42)

In [ ]:
!pip install pulp

In [ ]:
DATA_PATH = 'sl_railway_ticketing_2020_2024.csv'
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

print('Shape:', df.shape)
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
print('Classes:', df['class'].unique())
df.head()

In [ ]:
# Weather is downloaded locally from Open-Meteo ERA5 archive for reproducible training.
# Station coordinates are OSM-derived; Ratnapura is intentionally unresolved.
WEATHER_PATH = 'weather/open_meteo_historical_station_weather_2020_2024.csv'
weather = pd.read_csv(WEATHER_PATH, parse_dates=['date'])

origin_weather = weather.rename(columns={c: f'origin_{c}' for c in weather.columns if c not in ['date', 'station']})
origin_weather = origin_weather.rename(columns={'station': 'origin_station'})
destination_weather = weather.rename(columns={c: f'destination_{c}' for c in weather.columns if c not in ['date', 'station']})
destination_weather = destination_weather.rename(columns={'station': 'destination_station'})

rows_before_weather_join = len(df)
data = df.merge(origin_weather, on=['date', 'origin_station'], how='inner')
data = data.merge(destination_weather, on=['date', 'destination_station'], how='inner')
print('Demand rows before weather join:', rows_before_weather_join)
print('Demand rows after weather join:', len(data))
print('Rows excluded due to unavailable weather:', rows_before_weather_join - len(data))
print('Weather stations in joined data:', sorted(set(data['origin_station']) | set(data['destination_station'])))

In [ ]:
daily_demand = df.groupby('date')['tickets_sold'].sum()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

daily_demand.rolling(30).mean().plot(ax=axes[0,0], title='30-Day Rolling Avg - Total Daily Demand')
axes[0,0].set_ylabel('Tickets sold')

df.groupby('class')['tickets_sold'].sum().reindex(['1st Class','2nd Class','3rd Class']).plot(
    kind='bar', ax=axes[0,1], color=['#4C72B0','#55A868','#C44E52'])
axes[0,1].set_title('Total Demand by Travel Class')
axes[0,1].set_xticklabels(axes[0,1].get_xticklabels(), rotation=0)

df.groupby(df['date'].dt.month)['tickets_sold'].mean().plot(kind='bar', ax=axes[1,0], color='#8172B2')
axes[1,0].set_title('Average Demand per Record by Month (Seasonality)')
axes[1,0].set_xlabel('Month')
axes[1,0].tick_params(axis='x', rotation=0)

df.groupby('is_public_holiday')['seat_utilization_pct'].mean().plot(kind='bar', ax=axes[1,1], color=['#999999','#DD8452'])
axes[1,1].set_title('Avg Seat Utilization: Holiday vs Non-Holiday')
axes[1,1].set_xticklabels(['Non-Holiday','Holiday'], rotation=0)

plt.tight_layout()
plt.savefig('outputs/eda_overview.png', dpi=120)
plt.show()

In [ ]:
# Date and calendar features are known on the prediction date.
data['dayofweek'] = data['date'].dt.dayofweek
data['day'] = data['date'].dt.day
data['weekofyear'] = data['date'].dt.isocalendar().week.astype(int)
data['is_weekend'] = (data['dayofweek'] >= 5).astype(int)
data['month_sin'] = np.sin(2*np.pi*data['month']/12)
data['month_cos'] = np.cos(2*np.pi*data['month']/12)
data['dow_sin'] = np.sin(2*np.pi*data['dayofweek']/7)
data['dow_cos'] = np.cos(2*np.pi*data['dayofweek']/7)
data['route'] = data['origin_station'] + ' - ' + data['destination_station']

# Route weather is joined on the same date as demand; no future weather is used.
data['average_temperature'] = (data['origin_temperature_2m_mean'] + data['destination_temperature_2m_mean']) / 2
data['total_route_precipitation'] = data['origin_precipitation_sum'] + data['destination_precipitation_sum']
data['total_route_rain'] = data['origin_rain_sum'] + data['destination_rain_sum']
data['total_route_precipitation_hours'] = data['origin_precipitation_hours'] + data['destination_precipitation_hours']
data['average_route_wind_speed'] = (data['origin_wind_speed_10m_max'] + data['destination_wind_speed_10m_max']) / 2

data = data.sort_values(['route', 'train_type', 'class', 'date'])
group_columns = ['route', 'train_type', 'class']
grouped_demand = data.groupby(group_columns)['tickets_sold']

# Every feature below uses only observations strictly before the current row.
data['lag_1'] = grouped_demand.shift(1)
data['lag_7'] = grouped_demand.shift(7)
prior_demand = grouped_demand.shift(1)
data['roll_mean_7'] = prior_demand.groupby([data[c] for c in group_columns]).transform(lambda s: s.rolling(7, min_periods=1).mean())
data['roll_mean_30'] = prior_demand.groupby([data[c] for c in group_columns]).transform(lambda s: s.rolling(30, min_periods=1).mean())

# Early records use only prior observations: missing lag_1 becomes 0, while lag_7
# uses the expanding mean of prior demand. No complete-series/global mean is used.
prior_mean = prior_demand.groupby([data[c] for c in group_columns]).transform(lambda s: s.expanding(min_periods=1).mean())
data['lag_1'] = data['lag_1'].fillna(0)
data['lag_7'] = data['lag_7'].fillna(prior_mean).fillna(0)
data[['roll_mean_7', 'roll_mean_30']] = data[['roll_mean_7', 'roll_mean_30']].fillna(0)

cat_cols = ['route', 'train_type', 'class', 'line_name', 'day_type']
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    data[c + '_enc'] = le.fit_transform(data[c])
    encoders[c] = le

feature_cols = [
    'route_enc', 'train_type_enc', 'class_enc', 'line_name_enc', 'day_type_enc',
    'distance_km', 'seat_capacity', 'is_public_holiday',
    'month', 'dayofweek', 'day', 'weekofyear', 'is_weekend',
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    'lag_1', 'lag_7', 'roll_mean_7', 'roll_mean_30',
    'average_temperature', 'total_route_precipitation', 'total_route_rain',
    'total_route_precipitation_hours', 'average_route_wind_speed'
]
target_col = 'tickets_sold'
print('Feature matrix shape:', data[feature_cols].shape)
print('COVID is not in model features:', 'is_covid_lockdown_period' not in feature_cols)
data[feature_cols + [target_col, 'class']].head()

In [ ]:
train_mask = data['date'] < '2024-01-01'
test_mask  = data['date'] >= '2024-01-01'

X_train, y_train = data.loc[train_mask, feature_cols], data.loc[train_mask, target_col]
X_test,  y_test  = data.loc[test_mask, feature_cols],  data.loc[test_mask, target_col]

print('Train rows:', X_train.shape[0], '| Test rows:', X_test.shape[0])

In [ ]:
models = {
    'RandomForest': RandomForestRegressor(n_estimators=300, max_depth=14, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                             subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1)
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mape = np.mean(np.abs((y_test - preds) / y_test.replace(0, 1))) * 100
    print(f'{name:15s} | MAE: {mae:6.2f}  RMSE: {rmse:6.2f}  MAPE: {mape:5.2f}%')

# Pick the best model (lowest MAE) as the production forecaster
best_name = min(trained_models, key=lambda n: mean_absolute_error(y_test, trained_models[n].predict(X_test)))
best_model = trained_models[best_name]
print(f'\nSelected model: {best_name}')

In [ ]:
data.loc[test_mask, 'predicted_demand'] = best_model.predict(X_test)

class_metrics = []
for cls in ['1st Class', '2nd Class', '3rd Class']:
    sub = data.loc[test_mask & (data['class'] == cls)]
    mae = mean_absolute_error(sub[target_col], sub['predicted_demand'])
    rmse = np.sqrt(mean_squared_error(sub[target_col], sub['predicted_demand']))
    mape = np.mean(np.abs((sub[target_col] - sub['predicted_demand']) / sub[target_col].replace(0,1))) * 100
    class_metrics.append({'class': cls, 'MAE': round(mae,2), 'RMSE': round(rmse,2), 'MAPE_%': round(mape,2)})

class_metrics_df = pd.DataFrame(class_metrics)
class_metrics_df.to_csv('outputs/class_wise_metrics.csv', index=False)
class_metrics_df

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

daily = df.groupby('date')['tickets_sold'].sum().asfreq('D').ffill()

scaler = MinMaxScaler()
scaled = scaler.fit_transform(daily.values.reshape(-1,1))

LOOKBACK = 14
X_seq, y_seq, dates_seq = [], [], []
for i in range(LOOKBACK, len(scaled)):
    X_seq.append(scaled[i-LOOKBACK:i, 0])
    y_seq.append(scaled[i, 0])
    dates_seq.append(daily.index[i])
X_seq, y_seq = np.array(X_seq), np.array(y_seq)
X_seq = X_seq.reshape((X_seq.shape[0], X_seq.shape[1], 1))

split_idx = np.searchsorted(dates_seq, pd.Timestamp('2024-01-01'))
X_seq_train, X_seq_test = X_seq[:split_idx], X_seq[split_idx:]
y_seq_train, y_seq_test = y_seq[:split_idx], y_seq[split_idx:]
dates_test = np.array(dates_seq)[split_idx:]

lstm = keras.Sequential([
    layers.Input(shape=(LOOKBACK, 1)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])
lstm.compile(optimizer='adam', loss='mse')
history = lstm.fit(X_seq_train, y_seq_train, epochs=30, batch_size=16,
                    validation_split=0.1, verbose=0)

pred_scaled = lstm.predict(X_seq_test, verbose=0)
pred_actual = scaler.inverse_transform(pred_scaled).flatten()
true_actual = scaler.inverse_transform(y_seq_test.reshape(-1,1)).flatten()

lstm_mae = mean_absolute_error(true_actual, pred_actual)
lstm_rmse = np.sqrt(mean_squared_error(true_actual, pred_actual))
print(f'LSTM aggregate daily-demand forecast -> MAE: {lstm_mae:.1f}  RMSE: {lstm_rmse:.1f}')

actual_vs_pred = pd.DataFrame({
    'date': dates_test,
    'actual_demand': true_actual.round(0).astype(int),
    'predicted_demand': pred_actual.round(0).astype(int)
})
actual_vs_pred.to_csv('outputs/actual_vs_predicted.csv', index=False)

plt.figure(figsize=(12,4))
plt.plot(actual_vs_pred['date'], actual_vs_pred['actual_demand'], label='Actual')
plt.plot(actual_vs_pred['date'], actual_vs_pred['predicted_demand'], label='Predicted')
plt.title('Actual vs Predicted Total Daily Demand (2024, LSTM)')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/actual_vs_predicted.png', dpi=120)
plt.show()

In [ ]:
peak_dow = df.groupby('day_of_week')['tickets_sold'].mean().reindex(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
peak_month = df.groupby('month')['tickets_sold'].mean()

holiday_factor = (df.groupby('is_public_holiday')['seat_utilization_pct'].mean())
holiday_uplift_pct = round((holiday_factor[1] / holiday_factor[0] - 1) * 100, 1)

peak_dow.to_csv('outputs/peak_time_by_dayofweek.csv', header=['avg_tickets_sold'])
peak_month.to_csv('outputs/seasonal_factors_by_month.csv', header=['avg_tickets_sold'])

seasonal_holiday_summary = {
    'peak_day_of_week': peak_dow.idxmax(),
    'lowest_day_of_week': peak_dow.idxmin(),
    'peak_month': int(peak_month.idxmax()),
    'lowest_month': int(peak_month.idxmin()),
    'holiday_utilization_uplift_pct': holiday_uplift_pct
}
with open('outputs/seasonal_holiday_summary.json', 'w') as f:
    json.dump(seasonal_holiday_summary, f, indent=2)

print(seasonal_holiday_summary)

fig, ax = plt.subplots(1, 2, figsize=(13,4))
peak_dow.plot(kind='bar', ax=ax[0], color='#4C72B0', title='Peak Time Analysis - Avg Demand by Day of Week')
ax[0].tick_params(axis='x', rotation=45)
peak_month.plot(kind='bar', ax=ax[1], color='#55A868', title='Seasonal Factors - Avg Demand by Month')
plt.tight_layout()
plt.savefig('outputs/peak_seasonal_analysis.png', dpi=120)
plt.show()

In [ ]:
FAIRNESS_FLOOR = 0.70   # every class keeps at least 70% of its base capacity
FLEX_CAP       = 1.20   # a class can grow at most 20% beyond its base capacity

def allocate_seats_for_train(base_caps: dict, predicted_demand: dict):
    """base_caps, predicted_demand: {'1st Class': x, '2nd Class': y, '3rd Class': z}"""
    total_capacity = sum(base_caps.values())
    classes = list(base_caps.keys())

    prob = pulp.LpProblem('SeatAllocation', pulp.LpMaximize)
    alloc = {c: pulp.LpVariable(f'alloc_{c}', lowBound=0, cat='Integer') for c in classes}
    satisfied = {c: pulp.LpVariable(f'satisfied_{c}', lowBound=0, cat='Integer') for c in classes}

    prob += pulp.lpSum(satisfied.values())  # maximize total satisfied demand

    prob += pulp.lpSum(alloc.values()) == total_capacity
    for c in classes:
        prob += alloc[c] >= FAIRNESS_FLOOR * base_caps[c]
        prob += alloc[c] <= FLEX_CAP * base_caps[c]
        prob += satisfied[c] <= alloc[c]
        prob += satisfied[c] <= predicted_demand[c]

    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    return {c: int(alloc[c].value()) for c in classes}

# --- quick sanity check on one example train service ---
example_caps = {'1st Class': 37, '2nd Class': 225, '3rd Class': 487}
example_demand = {'1st Class': 60, '2nd Class': 150, '3rd Class': 500}
print('Example allocation:', allocate_seats_for_train(example_caps, example_demand))

In [ ]:
test_df = data.loc[test_mask, ['date','route','train_type','class','seat_capacity','predicted_demand', target_col]].copy()
test_df['predicted_demand'] = test_df['predicted_demand'].clip(lower=0).round().astype(int)

alloc_records = []
group_cols = ['date','route','train_type']
for (d, route, ttype), grp in test_df.groupby(group_cols):
    base_caps = dict(zip(grp['class'], grp['seat_capacity']))
    pred_dem  = dict(zip(grp['class'], grp['predicted_demand']))
    # ensure all 3 classes present
    if len(base_caps) < 3:
        continue
    allocation = allocate_seats_for_train(base_caps, pred_dem)
    for cls in base_caps:
        alloc_records.append({
            'date': d, 'route': route, 'train_type': ttype, 'class': cls,
            'base_capacity': base_caps[cls],
            'predicted_demand': pred_dem[cls],
            'allocated_seats': allocation[cls],
            'actual_tickets_sold': int(grp.loc[grp['class']==cls, target_col].values[0])
        })

alloc_df = pd.DataFrame(alloc_records)
alloc_df['seat_utilization_pct'] = (alloc_df[['allocated_seats','predicted_demand']].min(axis=1)
                                     / alloc_df['allocated_seats'] * 100).round(2)
alloc_df.to_csv('outputs/seat_allocation_results.csv', index=False)
print('Allocation rows generated:', len(alloc_df))
alloc_df.head(10)

In [ ]:
class_alloc_summary = alloc_df.groupby('class').agg(
    total_base_capacity=('base_capacity','sum'),
    total_allocated_seats=('allocated_seats','sum'),
    total_predicted_demand=('predicted_demand','sum'),
    avg_utilization_pct=('seat_utilization_pct','mean')
).reindex(['1st Class','2nd Class','3rd Class']).round(2)

class_alloc_summary.to_csv('outputs/class_wise_allocation_summary.csv')
class_alloc_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,4))
class_alloc_summary[['total_base_capacity','total_allocated_seats','total_predicted_demand']].plot(
    kind='bar', ax=axes[0], title='Class-wise Allocation vs Predicted Demand')
axes[0].tick_params(axis='x', rotation=0)

class_alloc_summary['avg_utilization_pct'].plot(kind='bar', ax=axes[1], color='#C44E52',
                                                 title='Seat Utilization % by Class')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylim(0,100)

plt.tight_layout()
plt.savefig('outputs/class_wise_allocation.png', dpi=120)
plt.show()

In [ ]:
dashboard_summary = {
    'total_forecasted_demand': int(alloc_df['predicted_demand'].sum()),
    'total_allocated_seats': int(alloc_df['allocated_seats'].sum()),
    'overall_seat_utilization_pct': round(
        alloc_df[['allocated_seats','predicted_demand']].min(axis=1).sum() / alloc_df['allocated_seats'].sum() * 100, 2
    ),
    'period_covered': f"{alloc_df['date'].min()} to {alloc_df['date'].max()}",
    'model_used_for_demand': best_name,
    'class_wise': class_alloc_summary.to_dict(orient='index')
}

with open('outputs/dashboard_summary.json', 'w') as f:
    json.dump(dashboard_summary, f, indent=2, default=str)

dashboard_summary

In [ ]:
demand_by_class_ts = test_df.groupby(['date','class'])[['predicted_demand', target_col]].sum().reset_index()
demand_by_class_ts = demand_by_class_ts.rename(columns={target_col: 'actual_demand'})
demand_by_class_ts.to_csv('outputs/demand_by_class_timeseries.csv', index=False)
demand_by_class_ts.head()

In [ ]:
import joblib
joblib.dump(best_model, 'outputs/demand_forecast_model.joblib')
joblib.dump(encoders, 'outputs/label_encoders.joblib')
lstm.save('outputs/lstm_aggregate_demand.keras')
print('Saved: demand_forecast_model.joblib, label_encoders.joblib, lstm_aggregate_demand.keras')

In [ ]:
# Prediction-time feature construction mirrors training: date/calendar, leakage-safe lags,
# endpoint weather from the live service, and the same five route-level weather features.
import joblib
loaded_model = joblib.load('outputs/demand_forecast_model.joblib')
loaded_encoders = joblib.load('outputs/label_encoders.joblib')

feature_cols = [
    'route_enc', 'train_type_enc', 'class_enc', 'line_name_enc', 'day_type_enc',
    'distance_km', 'seat_capacity', 'is_public_holiday', 'month', 'dayofweek',
    'day', 'weekofyear', 'is_weekend', 'month_sin', 'month_cos', 'dow_sin',
    'dow_cos', 'lag_1', 'lag_7', 'roll_mean_7', 'roll_mean_30',
    'average_temperature', 'total_route_precipitation', 'total_route_rain',
    'total_route_precipitation_hours', 'average_route_wind_speed'
]

# The live service resolves weather automatically from the verified station mapping.
# This example demonstrates the final model contract without inventing weather values.
new_service = pd.DataFrame([{
    'date': '2025-01-15', 'origin_station': 'Colombo Fort', 'destination_station': 'Kandy',
    'line_name': 'Main Line', 'train_type': 'Intercity', 'class': cls,
    'distance_km': 121, 'seat_capacity': {'1st Class': 37, '2nd Class': 225, '3rd Class': 487}[cls],
    'is_public_holiday': 0, 'day_type': 'Weekday'
} for cls in ['1st Class', '2nd Class', '3rd Class']])
print('Live prediction uses the feature contract:', feature_cols)

## Weather, leakage prevention, and COVID removal

Lag and rolling demand features are used because recent route/train/class demand contains useful temporal information. Every feature is shifted so the current demand target and all future observations are excluded. `lag_1` uses the previous available observation; `lag_7` uses the observation seven available periods earlier when present and otherwise uses an expanding mean calculated only from prior observations; rolling means use only the preceding observations. If no prior observation exists, the initialization is zero. No global or complete-series mean is used.

The training split is chronological: 2020-01-01 through 2023-12-31 for training and 2024-01-01 through 2024-12-31 for testing. Weather is joined by the same date and both route endpoints using local Open-Meteo ERA5 data. Ratnapura remains unresolved, so the 3,675 Colombo Fort to Ratnapura records are excluded: 51,627 of 55,302 records remain (6.6453% excluded). COVID remains only in the raw source for provenance and is absent from model features and prediction construction. Historical weather is reanalysis/model data, not direct railway-station observations.